In [ ]:
UNI_RANDOM_SEED = 2024
DEVICE = 0

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

torch.cuda.set_device(DEVICE)

import pdb
import pickle as pkl
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

try:
    import open3d
    from visual_utils import open3d_vis_utils as V
    OPEN3D_FLAG = True
except:
    import mayavi.mlab as mlab
    from visual_utils import visualize_utils as V
    OPEN3D_FLAG = False
    
from raytorch.LiDAR import LiDAR_base
from cudaext.ops.Rotated_IoU.oriented_iou_loss import cal_iou_3d, assign_target_3d

from pytorch3d.vis.plotly_vis import plot_scene

from pcdet.datasets.kitti.kitti_dataset import create_kitti_infos
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import KittiDataset, build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

from data_tools import adv_dataset, kitti_carla_dataset
from eval_utils import eval_utils
from loss_utils import mesh_objectwise_loss, relevant_bounding_box_loss
from optim_utils import objectwise_deepfool

EVAL_OUTPUT_DIR = "./eval_output/"
CFG_FILE = "./cfgs/kitti_models/pointrcnn.yaml"
DATA_CONFIG_FILE = "./cfgs/dataset_configs/kitti_dataset.yaml"
DATA_PATH = "/home/ksas/Public/datasets/KITTI"
CKPT_PATH = "/home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth"
ROOFTOP_ANNOTATE = "/home/ksas/uzuki_space/vehicle-shape-reconstruction/rooftop_appro_std.pkl"
# ROOFTOP_ANNOTATE = None

BATCH_SIZE = 1
WORKERS = 4
DIST_TEST = False

OPTIM = "rbboxloss"
EVAL_INIT_PATCH = True

CHECK_GRAD_QUAD = False

cfg_from_yaml_file(CFG_FILE, cfg)

# BATCH_SIZE = cfg.OPTIMIZATION.BATCH_SIZE_PER_GPU
logger = common_utils.create_logger()
logger.info('-----------------Kitti Attack Test-------------------------')

roi_head_weights = 1.0
laplacian_weights = 0.001
learning_rate = 0.005

In [ ]:
def roipooling_grad_mapping(pooled_features_grad, batch_point_features, pooled_pts_idx):
    
    batch_size = batch_point_features.size(0)
    npoint = batch_point_features.size(1)
    feature_size = batch_point_features.size(2)
    
    batch_point_features_grad = torch.zeros_like(batch_point_features)
    xyz_features_grad = batch_point_features.new_zeros((batch_size, npoint, 3))
    
    for batch_mask in range(0, batch_size):
        pts_idx_expanded = pooled_pts_idx[batch_mask].view(-1).long().unsqueeze(1).expand(-1, feature_size)
        xyz_pooled_features_grad_viewed = pooled_features_grad[:, :, :3].view(-1, 3)
        pooled_features_grad_viewed = pooled_features_grad[:, :, 3:].view(-1, feature_size)

        batch_point_features_grad[batch_mask].scatter_add_(0, pts_idx_expanded, pooled_features_grad_viewed)
        xyz_features_grad[batch_mask].scatter_add_(0, pts_idx_expanded[:, :3], xyz_pooled_features_grad_viewed)
    
    return xyz_features_grad, batch_point_features_grad

def gtbox_wise_cos_compute(mesh_proposal_loss:torch.Tensor, 
                           gtbox_idx:torch.Tensor,
                           gtbox_size:int,
                           optimizer,
                           universal_adv_patch):
    grad_list = []
    for idx in range(gtbox_size):
        gtbox_mask = (gtbox_idx == idx)
        masked_loss = mesh_proposal_loss[gtbox_mask]
        optimizer.zero_grad()
        masked_loss.sum().backward(retain_graph = True)
        grad_list.append(universal_adv_patch.get_mesh_gradient())
    
    # for i in range(gtbox_size):
    #     for j in range(gtbox_size - i - 1):
            
    # pdb.set_trace()

rooftop_approximate = None
try:
    with open(ROOFTOP_ANNOTATE, "rb") as input:
        rooftop_approximate = pkl.load(input)
except FileNotFoundError as error:
    logger.info(error.__str__())
except TypeError as error:
    logger.info(error.__str__())
    
logger.info(f"rooftop_approximate: {rooftop_approximate}")

In [ ]:
test_set, test_loader, sampler = build_dataloader(
        dataset_cfg=cfg.DATA_CONFIG,
        class_names=cfg.CLASS_NAMES,
        batch_size=BATCH_SIZE,
        dist=DIST_TEST, workers=WORKERS, logger=logger, training=False
    )
logger.info(f'Class names of samples: \t{test_set.class_names}')

2024-04-02 11:45:54,963   INFO  ground truth boxes statistic: [    0 28742  4487  1627]

In [ ]:
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=test_set)
model.load_params_from_file(filename=CKPT_PATH, logger=logger, to_cpu=True)
model.cuda()
model.eval()

for idx, module in enumerate(model.module_list):
    logger.info(f'Module names of model \t({idx}): \t{module._get_name()}')
    
backbone_network = model.module_list[0]
point_headbox = model.module_list[1]
pointrcnn_head = model.module_list[2]

lidar = LiDAR_base(origin=torch.tensor([0.0, 0.0, 0.0]).cuda(),
                   azi_range=[-90, 90],
                   polar_range= [-2.18, 2.0],
                   polar_num=10, azi_res=0.08)
kitti_adv_dataset = adv_dataset(test_set,
                                sample_amount=[50, 25],
                                rooftop_approximate = rooftop_approximate,
                                surrogate_model=None,
                                lidar = lidar,
                                enable_car = True,
                                enable_ped = False,
                                enable_bicycle = False)

optimizer = optim.Adam(kitti_adv_dataset.get_adversarial_parameter(), 
                       lr=learning_rate)

In [ ]:
if EVAL_INIT_PATCH:
    kitti_adv_dataset.enable_adversarial_patch(True)
    eval_utils.eval_one_epoch(
            cfg, None, model, kitti_adv_dataset, 0, logger, dist_test=DIST_TEST,
            result_dir=Path(EVAL_OUTPUT_DIR), 
            infer_time=True
        )

Pedestrian AP@0.50, 0.50, 0.50:
bbox AP:77.2540, 72.7474, 67.2434
bev  AP:76.5531, 68.1945, 64.7444
3d   AP:75.7239, 67.4655, 63.2433
aos  AP:76.44, 71.66, 66.12

Pedestrian AP@0.50, 0.50, 0.50:
bbox AP:40.1128, 32.2185, 30.1760
bev  AP:38.7276, 30.6786, 28.5188
3d   AP:36.7903, 29.4880, 24.0223
aos  AP:38.06, 30.49, 28.49

Pedestrian AP@0.50, 0.50, 0.50:
bbox AP:40.1037, 32.5085, 29.9554
bev  AP:38.3680, 30.6932, 28.4373
3d   AP:36.5991, 29.4193, 24.2488
aos  AP:38.27, 30.97, 28.48

In [ ]:
car_rbbox_loss_func = relevant_bounding_box_loss(frozen_iou = False,
                 frozen_logit = False,
                 confidence_threshold = 0.1,
                 iou_threshold = 0.1, 
                 verbose = True)

ped_rbbox_loss_func = relevant_bounding_box_loss(frozen_iou = False,
                 frozen_logit = False,
                 confidence_threshold = 0.01,
                 iou_threshold = 0.01, 
                 verbose = True)

grad_cache = []

def evaluate_one_epoch_attack(enable_adv, update, visualize, verbose_epoch: int = 100):
    kitti_adv_dataset.enable_adversarial_patch(enable_adv)
    
    for i, batch_dict in tqdm(enumerate(kitti_adv_dataset), total=kitti_adv_dataset.__len__()):
        load_data_to_gpu(batch_dict)

        if not torch.eq(batch_dict['gt_boxes'][0, :, 7], 1).any() and\
           not torch.eq(batch_dict['gt_boxes'][0, :, 7], 2).any() :
            # logger.info(f"no vehicles found in batch \t{i}")
            continue
        
        model.eval()
        model.zero_grad()
        pred_dicts, _ = model(batch_dict)
        point_headbox_ret_dict = point_headbox.forward_ret_dict
        pointrcnn_head_ret_dict = pointrcnn_head.forward_ret_dict
        
        if CHECK_GRAD_QUAD:
            n_size = batch_dict["gt_boxes"].size(1)
            grad_cache.append([])
            for n_idx_mask in range(n_size):
                mesh_proposal_loss:torch.Tensor = rbbox_loss_func(batch_dict = point_headbox_ret_dict, 
                                        gt_boxes = batch_dict["gt_boxes"][:, n_idx_mask:n_idx_mask+1, :], 
                                        target_class = 1,
                                        logit_normal = "sigmoid",
                                        ret_part_loss = False)

                optimizer.zero_grad()
                model.zero_grad()
                if mesh_proposal_loss.requires_grad:
                    mesh_proposal_loss.backward(retain_graph = True)
                    grad_cache[-1].append([grad.cpu().numpy() for grad in kitti_adv_dataset.universal_adv_patch.get_mesh_gradient()])
            
        if OPTIM == "rbboxloss":
            car_mesh_proposal_loss = car_rbbox_loss_func(batch_dict = point_headbox_ret_dict, 
                                    gt_boxes = batch_dict["gt_boxes"], 
                                    target_class = 1,
                                    logit_normal = "sigmoid",
                                    ret_part_loss = False)
            
            ped_mesh_proposal_loss = ped_rbbox_loss_func(batch_dict = point_headbox_ret_dict, 
                                    gt_boxes = batch_dict["gt_boxes"], 
                                    target_class = 2,
                                    logit_normal = "sigmoid",
                                    ret_part_loss = False)
            
            # mesh_proposal_loss, gtbox_idx, gtbox_size = rbbox_loss_func(batch_dict = point_headbox_ret_dict, 
            #                         gt_boxes = batch_dict["gt_boxes"], 
            #                         target_class = 1,
            #                         logit_normal = "sigmoid",
            #                         return_gtbox_id = True,
            #                         ret_part_loss = False)
            
            # mesh_head_loss = rbbox_loss_func(batch_dict = pointrcnn_head_ret_dict, 
            #                         gt_boxes = batch_dict["gt_boxes"], 
            #                         target_class = 1,
            #                         logit_normal = "sigmoid",
            #                         input_type = 'roihead',
            #                         ret_part_loss = False)
            
            # gtbox_wise_cos_compute(mesh_proposal_loss = mesh_proposal_loss, 
            #                gtbox_idx = gtbox_idx,
            #                gtbox_size = gtbox_size,
            #                optimizer = optimizer,
            #                universal_adv_patch = kitti_adv_dataset.universal_adv_patch)
            
            # if mesh_proposal_loss.size(0) != 1:
            # mesh_proposal_loss = mesh_proposal_loss.sum()
                
            regular_loss = kitti_adv_dataset.universal_adv_patch_car.get_laplacian_loss()\
                            + kitti_adv_dataset.universal_adv_patch_ped.get_laplacian_loss()
            # total_loss = mesh_proposal_loss + roi_head_weights * mesh_head_loss + laplacian_weights * regular_loss
            total_loss = car_mesh_proposal_loss + ped_mesh_proposal_loss + laplacian_weights * regular_loss
            optimizer.zero_grad()
            model.zero_grad()
            total_loss.backward()
            # if pointrcnn_head.pooled_features.grad is not None:
            #     xyz_features_grad, batch_point_features_grad = roipooling_grad_mapping(pointrcnn_head.pooled_features.grad, 
            #                                                                         pointrcnn_head.batch_point_features, 
            #                                                                         pointrcnn_head.pooled_pts_idx)
            #     pointrcnn_head.batch_point_features.backward(batch_point_features_grad, retain_graph = True)
            #     batch_dict['points'][None, :, 1:4].backward(xyz_features_grad, retain_graph = True)
            
        else:
            raise NotImplementedError
        
        if verbose_epoch > 0 and i % verbose_epoch == 0:
            # logger.info(f"deformed verts of mesh: \t{kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert()}")
            # logger.info(f"deformed vert gradients of mesh: \t{kitti_adv_dataset.universal_adv_patch.get_mesh_gradient()}")

            if visualize:
                # fig = plot_scene({
                #     "original": {
                #         "mesh_1": kitti_adv_dataset.universal_adv_patch.get_basic_mesh()
                #     },
                #     "adversarial": {
                #         "mesh_1": kitti_adv_dataset.universal_adv_patch.get_deformed_mesh()
                #     },
                # }, ncols=2)
                # fig.update_layout(height=400, width=800)
                # fig.show()
                
                V.draw_scenes(
                    points=batch_dict['points'][:, 1:], ref_boxes=pred_dicts[0]['pred_boxes'].detach(),
                    ref_scores=pred_dicts[0]['pred_scores'].detach(), ref_labels=pred_dicts[0]['pred_labels'].detach(), gt_boxes=batch_dict['gt_boxes'][0]
                )
            
        if update:
            """
                set grad along z-axi to 0.
            """
            vert_grad, translate_grad, theta_grad = kitti_adv_dataset.universal_adv_patch_car.get_mesh_gradient()
            vert_grad[:, 2] = 0.
            translate_grad[2] = 0.
            
            vert_grad, translate_grad, theta_grad = kitti_adv_dataset.universal_adv_patch_ped.get_mesh_gradient()
            vert_grad[:, 2] = 0.
            translate_grad[2] = 0.
            
            if OPTIM == "rbboxloss":
                optimizer.step()
            else:
                raise NotImplementedError
            
evaluate_one_epoch_attack(enable_adv = True, 
                            update = True, 
                            visualize = True, 
                            verbose_epoch= 1)

with open("./check_grad.pkl", "wb") as output:
    pkl.dump(grad_cache, output)

In [ ]:
fig = plot_scene({
    "original (car)": {
        "mesh_1": kitti_adv_dataset.universal_adv_patch_car.get_basic_mesh()
    },
    "adversarial (car)": {
        "mesh_1": kitti_adv_dataset.universal_adv_patch_car.get_deformed_mesh()
    },
    "original (ped)": {
        "mesh_1": kitti_adv_dataset.universal_adv_patch_ped.get_basic_mesh()
    },
    "adversarial (ped)": {
        "mesh_1": kitti_adv_dataset.universal_adv_patch_ped.get_deformed_mesh()
    },
    
}, ncols=2)
fig.update_layout(height=800, width=800)
fig.show()

logger.info(f"deformed verts of mesh: \t{kitti_adv_dataset.universal_adv_patch_car.get_mesh_deform_vert()}")
logger.info(f"theta of mesh: \t{kitti_adv_dataset.universal_adv_patch_car.theta}")
logger.info(f"theta of global_translation: \t{kitti_adv_dataset.universal_adv_patch_car.global_translation}")

In [ ]:
kitti_adv_dataset.enable_adversarial_patch(True)
eval_utils.eval_one_epoch(
        cfg, None, model, kitti_adv_dataset, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR), 
        infer_time=True
    )